# DOMAIN ADAPTATION

In this notebook, we first explore domain gap between public and local data. Initially, we have models trained in public data for classification of 4 leaves affections classes.



*   In Section 1, we will explore this gap, measuring $\Delta{F_1} = ({F_1})_{P} - ({F_1})_L $, where $P$ and $L$ refers to public and local test data, respectively.
*  Then we will use this as baseline, and in Section 2 we will explore 3 different fine-tuning techniques in order to adapt our models to local domain, enhancing it generalizing power.


*   Finally, in Section 3 we will discuss the best strategy, and check whether the models improve their adaptation capacity.


In [ ]:
# Core libraries
from pathlib import Path

import numpy as np
import pandas as pd
import cv2

import tensorflow as tf 
 
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback

import albumentations as A

# Scikit-learn metrics & utilities

from sklearn.metrics import (f1_score,
    accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight


from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam

# custom functions
from utils import *

from ultralytics import YOLO

# Transfer Learning for domain shift.

## CNN models

In [ ]:
BASE_PATH = Path("../data")
EXP_PATH = BASE_PATH / "EXPERIMENT1"
DOMAIN_PATH = EXP_PATH / "domain_adaptation"

TRAIN_PATH = DOMAIN_PATH / "train_public_64" # CHANGE
VAL_PATH   = DOMAIN_PATH / "val_public_16"  # CHANGE
TEST_PUBLIC_PATH = DOMAIN_PATH / "test"   # CHANGE
TEST_LOCAL_PATH  = EXP_PATH / "local_data"          # CHANGE
MODELS_PATH = Path("../models")

BASE_MODELS_PATH = MODELS_PATH / "base_models"

FT_MODELS_PATH = MODELS_PATH / "finetuned_models"
FT_MODELS_PATH.mkdir(exist_ok=True)

BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_list = [
    "efficientnetb0_CBAM",
    "mobilenetv2_CBAM",
    "efficientnetb3_CBAM",
    "resnet101_CBAM",
    "mobilenetv2",
    "efficientnetb0",
    "mobilenetv3large",
    "densenet121",
    "efficientnetb3",
    "resnet50",
    "inceptionv3",
    "resnet101",
    "vgg16",
    "vgg19",
    "efficientnetb5"
]

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

CUSTOM_AUGMENTATION = A.Compose([
    A.Rotate(limit=10, border_mode=cv2.BORDER_CONSTANT, fill=0, p=0.5),
    A.RandomResizedCrop(
        size=(IMG_SIZE[0], IMG_SIZE[1]),
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=cv2.INTER_LINEAR,
        p=0.3
    ),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),

    A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.08, p=0.3),

    A.HueSaturationValue(hue_shift_limit=3, sat_shift_limit=5, val_shift_limit=4, p=0.2),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.05,
        rotate_limit=0,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        p=0.3
    ),

    A.Resize(width=IMG_SIZE[1], height=IMG_SIZE[0])
], p=1.0)

In [ ]:
def apply_fine_tuning_strategy(model, strategy):
    """
    Modifies the fine-tuning strategy depending of each case (3 options).
    """
    if strategy == "feature_extractor":
        # Not training backbone
        for layer in model.layers:
            layer.trainable = False

        # Just head
        if hasattr(model.layers[-1], "trainable"):
            model.layers[-1].trainable = True

    elif strategy == "fine_tune_last":
        # Unfreeze last 20 layers
        for layer in model.layers[:-20]:
            layer.trainable = False
        for layer in model.layers[-20:]:
            layer.trainable = True

    elif strategy == "fine_tune_all":
        # Unfreeze all
        for layer in model.layers:
            layer.trainable = True

    return model

In [ ]:

STRATEGIES = ["feature_extractor","fine_tune_last", "fine_tune_all"]
RESULTS = []

for backbone_name in model_list:
    print(f"\n" + "="*60 + f"\n🔹 PROCESSING: {backbone_name}\n" + "="*60)


    is_cbam = "_CBAM" in backbone_name
    alb_backbone = backbone_name.replace("_CBAM", "").lower()
    
    gen_params = {'batch_size': BATCH_SIZE, 'img_size': IMG_SIZE, 'backbone': alb_backbone}
    train_gen = AlbumentationsDataGenerator(EXP_PATH/"train", oversample=True, min_per_class=400, augment=CUSTOM_AUGMENTATION, **gen_params)
    val_gen   = AlbumentationsDataGenerator(EXP_PATH/"val", oversample=False, augment=False, **gen_params)
    test_gen  = AlbumentationsDataGenerator(EXP_PATH/"test", oversample=False, augment=False, **gen_params)

    # Pesos de clase
    y_train = train_gen.get_labels()
    cw_array = np.sqrt(compute_class_weight("balanced", classes=np.unique(y_train), y=y_train))
    cw_dict = {i: w for i, w in enumerate(cw_array)}

    # Cargar modelo con objetos personalizados dinámicos
    model_path = BASE_MODELS_PATH / f"{backbone_name}_best_model.keras"
    if not model_path.exists():
        print(f"Skipping: {model_path}, it does not exist."); continue

    custom_objects = {
    "weighted_categorical_crossentropy": weighted_categorical_crossentropy,
    "recall_macro": recall_macro,
    "precision_macro": precision_macro,
    "f1_macro": f1_macro
    }   

    base_model = load_model(model_path, compile=False, custom_objects=custom_objects)

    # for each strategy
    for strategy in STRATEGIES:
        print(f"\n--- STRATEGY: {strategy} ---")
        

        model_ft = tf.keras.models.clone_model(base_model)
        model_ft.set_weights(base_model.get_weights())
        model_ft = apply_fine_tuning_strategy(model_ft, strategy)

       
        loss_fn = weighted_categorical_crossentropy(cw_array)

        model_ft.compile(
            optimizer=Adam(1e-4),
            loss=loss_fn,
            metrics=["accuracy", precision_macro, recall_macro, f1_macro]
        )

        strategy_dir = FT_MODELS_PATH / strategy
        strategy_dir.mkdir(parents=True, exist_ok=True)
        save_path = strategy_dir / f"{backbone_name}_{strategy}.keras"

        callbacks = [
            ModelCheckpoint(save_path, save_best_only=True, monitor="val_f1_macro", mode="max"),
            EarlyStopping(monitor="val_f1_macro", patience=5, restore_best_weights=True, mode="max"),
            ReduceLROnPlateau(monitor="val_f1_macro", factor=0.5, patience=2, min_lr=1e-6, mode="max"),
            ClassMetricsCallback(val_gen, class_names=train_gen.classes)
        ]

        model_ft.fit(
            train_gen, validation_data=val_gen, epochs=15,
            callbacks=callbacks, class_weight=cw_dict, verbose=1
        )

        y_true, y_pred = [], []
        for i in range(len(test_gen)):
            Xb, yb = test_gen[i]
            y_true.extend(np.argmax(yb, axis=1))
            y_pred.extend(np.argmax(model_ft.predict(Xb, verbose=0), axis=1))

        f1_test = f1_score(y_true, y_pred, average="weighted")
        acc_test = accuracy_score(y_true, y_pred)
        
        print(f"✅ {backbone_name} ({strategy}): F1={f1_test:.4f} | Acc={acc_test:.4f}")

        RESULTS.append({
            "model": backbone_name, "strategy": strategy, "is_cbam": is_cbam,
            "f1_test": f1_test, "acc_test": acc_test, "path": str(save_path)
        })

df_results = pd.DataFrame(RESULTS)
display(df_results.sort_values(by="f1_test", ascending=False))

## YOLOV11 

In [ ]:
# Paths
DATASET_ROOT = EXP_PATH  # train/val/test folders there

models_info = {
    "yolov11n": BASE_MODELS_PATH / "YOLO11n_cls" / "weights" / "best.pt",
    "yolov11m": BASE_MODELS_PATH / "YOLO11m_cls" / "weights" / "best.pt",
    "yolov11l": BASE_MODELS_PATH / "YOLO11l_cls" / "weights" / "best.pt",
}


# strategies
strategies = {
    "feature_extractor": 10,   # freeze almost all
    "fine_tune_last": 5,       # freeze first layers
    "fine_tune_all": 0          # no freeze
}

# Fine-tuning loop
for strategy_name, freeze_value in strategies.items():
    print(f"\n=== Estrategia: {strategy_name} (freeze={freeze_value}) ===")


    strategy_dir = FT_MODELS_PATH / strategy_name
    strategy_dir.mkdir(parents=True, exist_ok=True)

    for model_name, model_path in models_info.items():
        print(f"\n>>> Fine-tuning modelo {model_name}")

        model = YOLO(model_path)

        # custom training using strategy
        model.train(
            data=DATASET_ROOT,
            epochs=15,
            imgsz=224,
            batch=32,
            device=0,
            project=strategy_dir,
            name=model_name,
            workers=4,
            freeze=freeze_value
        )

        print(f"✅ {model_name} fine-tune terminado con estrategia {strategy_name}")